# AI Avatar Studio — MiniMax H3 Colab GPU Worker (L4 / G4 版)

此 Notebook 整合自 [dseditor/h3-colab-skill](https://github.com/dseditor/h3-colab-skill)，透過 **ComfyUI + MiniMax H3** 提供電影級人物動態與聲音克隆。

> **硬體需求**：需 **Google Colab Pro / AI Pro**，並將執行階段切換為 **L4 GPU（或 G4 GPU）**（免費 T4 顯存不足以執行 MiniMax H3）。

In [ ]:
#@title 1. 檢查 GPU、拉取 h3-colab-skill 與安裝 ComfyUI
!nvidia-smi
!pip install -q fastapi uvicorn python-multipart httpx websockets yt-dlp
!apt-get -qq update && apt-get -qq install -y ffmpeg git-lfs
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb

import os
if not os.path.exists("/content/h3-colab-skill"):
    !git clone -q https://github.com/dseditor/h3-colab-skill.git /content/h3-colab-skill
if not os.path.exists("/content/ComfyUI"):
    !git clone -q https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
    !pip install -q -r /content/ComfyUI/requirements.txt

print("✅ 基礎環境與 h3-colab-skill 準備完成")

In [ ]:
#@title 2. 啟動 ComfyUI 並使用 h3-colab-skill 安裝 MiniMax H3 節點與模型 (首次約 5~8 分鐘)
import subprocess, sys, time, httpx
sys.path.insert(0, "/content/h3-colab-skill/h3-multimodal-colab/scripts")

# 先背景啟動 ComfyUI 讓 h3local.py setup 可以連線檢查與安裝節點
subprocess.run(["pkill", "-f", "/content/ComfyUI/main.py"], check=False)
subprocess.Popen(
    ["python", "/content/ComfyUI/main.py", "--listen", "127.0.0.1", "--port", "8188"],
    stdout=open("/content/comfyui.log", "w"),
    stderr=subprocess.STDOUT,
)
for _ in range(30):
    try:
        if httpx.get("http://127.0.0.1:8188/system_stats", timeout=2).status_code == 200:
            break
    except Exception:
        time.sleep(2)

!python /content/h3-colab-skill/h3-multimodal-colab/scripts/h3local.py --comfy http://127.0.0.1:8188 --comfy-dir /content/ComfyUI --models-dir /content/ComfyUI/models setup
print("✅ MiniMax H3 模型與 ComfyUI 節點設定完成")

In [ ]:
#@title 3. 啟動 ComfyUI + FastAPI Bridge + Cloudflare Tunnel
import json
import re
import subprocess
import sys
import threading
import time
import uuid
from pathlib import Path
import httpx
import torch
import uvicorn
from fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse

sys.path.insert(0, "/content/h3-colab-skill/h3-multimodal-colab/scripts")

API_TOKEN = ""  #@param {type:"string"}
WORK_DIR = Path("/content/h3_worker_tmp")
WORK_DIR.mkdir(parents=True, exist_ok=True)
COMFY_URL = "http://127.0.0.1:8188"

# 重啟 ComfyUI 確保新安裝的 Custom Nodes 已載入
subprocess.run(["pkill", "-f", "/content/ComfyUI/main.py"], check=False)
time.sleep(2)
subprocess.Popen(
    ["python", "/content/ComfyUI/main.py", "--listen", "127.0.0.1", "--port", "8188"],
    stdout=open("/content/comfyui.log", "w"),
    stderr=subprocess.STDOUT,
)
time.sleep(8)

app = FastAPI(title="Colab MiniMax H3 Worker")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

def verify_token(authorization: str | None):
    if API_TOKEN.strip() and authorization != f"Bearer {API_TOKEN.strip()}":
        raise HTTPException(status_code=401, detail="Invalid API token")

@app.get("/health")
def health(authorization: str | None = Header(default=None)):
    verify_token(authorization)
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    return {"status": "ok", "gpu": gpu_name, "engines": ["MiniMax-H3-TTS", "MiniMax-H3-Video"]}

@app.post("/api/tts")
async def h3_tts(
    text: str = Form(...),
    language: str = Form("zh"),
    speed: float = Form(1.0),
    ref_text: str = Form(""),
    ref_audio: UploadFile = File(...),
    engine: str = Form("minimax_h3"),
    authorization: str | None = Header(default=None),
):
    verify_token(authorization)
    req_id = uuid.uuid4().hex[:8]
    job_dir = WORK_DIR / f"tts_{req_id}"
    job_dir.mkdir(parents=True, exist_ok=True)
    ref_path = job_dir / "ref.wav"
    raw_ref = job_dir / f"raw_{ref_audio.filename}"
    raw_ref.write_bytes(await ref_audio.read())
    subprocess.run(["ffmpeg", "-y", "-i", str(raw_ref), "-t", "8", "-ar", "24000", "-ac", "1", str(ref_path)], check=True)

    manifest_path = job_dir / "manifest.json"
    manifest = {
        "jobs": [{
            "id": req_id,
            "type": "tts",
            "text": text.strip(),
            "ref_audio": "ref.wav",
        }]
    }
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False), encoding="utf-8")
    proc = subprocess.run(
        ["python", "/content/h3-colab-skill/h3-multimodal-colab/scripts/h3local.py", "--comfy", COMFY_URL, "--comfy-dir", "/content/ComfyUI", "--models-dir", "/content/ComfyUI/models", "run", str(manifest_path), "--confirmed"],
        capture_output=True,
        text=True,
    )
    wavs = list((job_dir / "out").glob("*.wav")) if (job_dir / "out").exists() else []
    if proc.returncode != 0 or not wavs:
        raise HTTPException(status_code=500, detail=f"MiniMax H3 TTS failed: {(proc.stdout + proc.stderr)[-600:]}")
    return FileResponse(str(wavs[0]), media_type="audio/wav", filename="h3_tts.wav")

@app.post("/api/lipsync")
async def h3_video(
    avatar: UploadFile = File(...),
    audio: UploadFile = File(...),
    engine: str = Form("minimax_h3"),
    authorization: str | None = Header(default=None),
):
    verify_token(authorization)
    req_id = uuid.uuid4().hex[:8]
    job_dir = WORK_DIR / f"vid_{req_id}"
    job_dir.mkdir(parents=True, exist_ok=True)
    ext = Path(avatar.filename or "face.png").suffix.lower() or ".png"
    face_name = f"face{ext}"
    (job_dir / face_name).write_bytes(await avatar.read())
    (job_dir / "audio.wav").write_bytes(await audio.read())

    manifest_path = job_dir / "manifest.json"
    manifest = {
        "jobs": [{
            "id": req_id,
            "type": "video",
            "drive_audio": "audio.wav",
            "ref_images": [face_name],
            "prompt": "A realistic presenter speaking naturally to the camera with expressive facial movement and accurate lip sync.",
        }]
    }
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False), encoding="utf-8")
    proc = subprocess.run(
        ["python", "/content/h3-colab-skill/h3-multimodal-colab/scripts/h3local.py", "--comfy", COMFY_URL, "--comfy-dir", "/content/ComfyUI", "--models-dir", "/content/ComfyUI/models", "run", str(manifest_path), "--confirmed"],
        capture_output=True,
        text=True,
    )
    mp4s = list((job_dir / "out").glob("*.mp4")) if (job_dir / "out").exists() else []
    if proc.returncode != 0 or not mp4s:
        raise HTTPException(status_code=500, detail=f"MiniMax H3 Video failed: {(proc.stdout + proc.stderr)[-600:]}")
    return FileResponse(str(mp4s[0]), media_type="video/mp4", filename="h3_result.mp4")

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8899, log_level="warning")

threading.Thread(target=run_server, daemon=True).start()
time.sleep(2)

print("🚀 正在啟動 MiniMax H3 Cloudflare Tunnel...")
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8899"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
for line in tunnel.stdout:
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        print("\n" + "=" * 60)
        print("✅ MiniMax H3 Colab Worker 已上線！")
        print(match.group(0))
        print("=" * 60 + "\n")